<a href="https://colab.research.google.com/github/Ankitravi1/ReelForge/blob/main/scripts/colab_wan2gp_server.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🎬 Autitic Studio — Google Colab Wan2GP & Z-Image GPU Server

Run remote GPU image & video generation for Autitic Studio on Google Colab (Free T4 or A100 GPU).

### What this provides:
1. **Wan 2.1 Video Generation** (Text-to-Video & Image-to-Video hero shots)
2. **Z-Image-Turbo / FLUX-Schnell** (Ultra-fast native 1024 stills in ~3-4s)
3. **Automatic Cloudflare Quick Tunnel** (Secure public HTTPS URL with no sign-up required)
4. **Google Drive Model Weight Cache** (Speeds up subsequent runs from 15 mins to 30 secs)

---

### Step 1: Verify GPU & Mount Google Drive (Optional for model weight caching)

In [ ]:
!nvidia-smi

# Set to True ONLY if your Google Drive has ample (>10GB) free space.
# Local Colab SSD (False) is recommended, has ~80GB free, and prevents Google Drive storage warnings!
CACHE_TO_DRIVE = False

if CACHE_TO_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    import os
    os.environ['HF_HOME'] = '/content/drive/MyDrive/reelforge_hf_cache'
    print('HuggingFace cache set to Google Drive: /content/drive/MyDrive/reelforge_hf_cache')
else:
    import os
    os.environ['HF_HOME'] = '/root/.cache/huggingface'
    print('✅ Using fast local Colab NVMe SSD cache (80 GB free, no Drive quota limits)')


### Step 2: Install Dependencies

In [ ]:
!pip install -q diffusers transformers accelerate fastapi uvicorn nest_asyncio imageio[ffmpeg] sentencepiece torchvision pydantic python-multipart

### Step 2.5 (Recommended): Pre-load Turbo Weights to GPU & Drive
Run this once to cache model weights into Google Drive (~45s). Subsequent generations will be instantaneous (~2.5s)!

In [ ]:
from diffusers import AutoPipelineForText2Image
import torch

print("📥 Pre-caching SDXL-Turbo weights to GPU / Google Drive...")
dtype = torch.float16 if torch.cuda.is_available() else torch.float32
pipe = AutoPipelineForText2Image.from_pretrained(
    "stabilityai/sdxl-turbo", torch_dtype=dtype, variant="fp16"
).to("cuda" if torch.cuda.is_available() else "cpu")
print("✅ SDXL-Turbo ready and cached in VRAM and Google Drive!")


### Step 3: Download and Launch the Autitic Studio Colab Server

In [ ]:
# 1. Stop any previously running server or tunnel
!pkill -f colab_gpu_server.py || true
!pkill -f cloudflared || true

import subprocess
import time
import re
import os

# 2. Download latest server script from GitHub
!wget -q -O colab_gpu_server.py https://raw.githubusercontent.com/Ankitravi1/ReelForge/main/scripts/colab_gpu_server.py

# 3. Download cloudflared if not present
!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared
!chmod +x cloudflared

# 4. Start the Autitic GPU Server on port 8000 with log streaming
print('Starting Autitic GPU Server on port 8000...')
log_file = open('colab_server.log', 'w')
server_proc = subprocess.Popen(['python', '-u', 'colab_gpu_server.py'], stdout=log_file, stderr=subprocess.STDOUT)
time.sleep(3)

# 5. Start Cloudflare Tunnel and grab the public URL
tunnel_proc = subprocess.Popen(
    ['./cloudflared', 'tunnel', '--url', 'http://localhost:8000'],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    universal_newlines=True
)

print('Connecting to Cloudflare...')
public_url = None
for line in iter(tunnel_proc.stdout.readline, ''):
    match = re.search(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', line)
    if match:
        public_url = match.group(0)
        print('\n' + '='*60)
        print(f'🚀 YOUR REELFORGE REMOTE GPU URL IS:\n{public_url}')
        print('='*60)
        print('\nPaste this URL into Autitic Studio -> Settings -> Remote GPU!')
        break

# 6. Keep tunnel active
try:
    tunnel_proc.wait()
except KeyboardInterrupt:
    print('Stopping...')
    tunnel_proc.terminate()
    server_proc.terminate()
    log_file.close()
